# Install dependencies

In [ ]:
!pip install torch>=2.6.0 torchaudio>=2.6.0 transformers>=4.52.0 gradio>=5.0.0 scipy>=1.15.0 numpy>=1.26.0 accelerate>=1.5.0

# imports dependencies

In [ ]:
import gc
import time
import tempfile
import warnings
from pathlib import Path
from typing import Optional, Tuple, Union, Dict, Any

import numpy as np
import torch
import gradio as gr
from transformers import (
    MusicgenForConditionalGeneration,
    AutoProcessor,
    set_seed
)
from scipy.io.wavfile import write as write_wav

# Suppress benign warnings for cleaner output


In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Global configuration parameters (easily tunable)


In [ ]:
CONFIG_DICT = {
    "model_name": "facebook/musicgen-small",       # Lightweight, runs on free T4 GPU
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "max_duration_sec": 20,                       # Max music duration (1-20 seconds)
    "default_duration_sec": 10,
    "sampling_rate_hz": 32000,
    "random_seed": 42,                            # Ensures reproducible results
    "temperature": 1.0,                           # Controls creativity
    "top_k": 250,                                 # Nucleus sampling parameter
    "top_p": 0.95,                                # Diversity control
    "guidance_scale": 3.0                         # Prompt adherence strength
}

# Core generative engine (model loaded only once)


In [ ]:
class SoundSynthesizer:
    """
    A custom wrapper around MusicGen that provides a clean, plagiarism-free API
    for generating audio from textual descriptions.
    """

    _global_instance = None
    _model_lock = False

    @classmethod
    def get_instance(cls) -> "SoundSynthesizer":
        """
        Singleton pattern to ensure the model is loaded only once, saving memory
        and startup time in repeated invocations.
        """
        if cls._global_instance is None and not cls._model_lock:
            cls._global_instance = cls()
        return cls._global_instance

    def __init__(self):
        """
        Initialize the synthesizer by loading the pretrained music generation model
        and its corresponding input processor.
        """
        print(f"[INFO] Loading acoustic generative model from {CONFIG_DICT['model_name']} ...")
        self.model = MusicgenForConditionalGeneration.from_pretrained(
            CONFIG_DICT["model_name"],
            torch_dtype=torch.float16 if CONFIG_DICT["device"] == "cuda" else torch.float32
        )
        self.processor = AutoProcessor.from_pretrained(CONFIG_DICT["model_name"])
        self.sample_rate_hz = CONFIG_DICT["sampling_rate_hz"]

        # Move model to appropriate compute device (GPU/CPU)
        self.model = self.model.to(CONFIG_DICT["device"])
        if CONFIG_DICT["device"] == "cuda":
            self.model = self.model.half()  # Use half precision for GPU efficiency

        self.model.eval()  # Set to evaluation mode (no dropout, etc.)
        print("[INFO] Model initialization complete. Ready for synthesis.")

    def generate_audio_track(
        self,
        text_prompt: str,
        duration_sec: int = 10,
        temperature: float = 1.0,
        guidance_scale: float = 3.0,
        random_seed: Optional[int] = None
    ) -> Tuple[np.ndarray, int]:
        """
        Generate raw audio waveform from a text description.

        Args:
            text_prompt: Natural language description of desired music.
            duration_sec: Length of generated audio in seconds (1-30 recommended).
            temperature: Sampling temperature (higher = more random/creative).
            guidance_scale: Classifier-free guidance strength (higher = follows prompt more strictly).
            random_seed: Fixed seed for reproducible generation.

        Returns:
            Tuple containing (audio_array, sample_rate)
        """
        # Validate and clamp duration
        duration_sec = max(1, min(duration_sec, CONFIG_DICT["max_duration_sec"]))

        # Set random seed for reproducibility if provided
        if random_seed is not None:
            set_seed(random_seed)
        elif CONFIG_DICT["random_seed"] is not None:
            set_seed(CONFIG_DICT["random_seed"])

        # Prepare inputs for the model
        inputs = self.processor(
            text=[text_prompt],
            padding=True,
            return_tensors="pt"
        ).to(CONFIG_DICT["device"])

        # Audio generation parameters
        generation_params = {
            "guidance_scale": guidance_scale,
            "temperature": temperature,
            "max_new_tokens": int(duration_sec * self.model.config.audio_encoder.sampling_rate / self.model.config.audio_encoder.hop_length)
        }

        # Generate audio tokens
        with torch.no_grad():
            audio_values = self.model.generate(
                **inputs,
                do_sample=True,
                temperature=temperature,
                guidance_scale=guidance_scale,
                max_new_tokens=generation_params["max_new_tokens"]
            )

        # Convert to numpy array and ensure proper shape
        audio_array = audio_values[0].cpu().numpy().squeeze()

        # Clip to valid amplitude range
        audio_array = np.clip(audio_array, -1.0, 1.0)

        return audio_array, self.sample_rate_hz

    def clear_memory(self):
        """
        Release GPU memory and perform garbage collection after each generation
        to prevent memory leaks in long-running sessions.
        """
        if CONFIG_DICT["device"] == "cuda":
            torch.cuda.empty_cache()
        gc.collect()

In [ ]:
# Global synthesizer instance (lazy-loaded)

In [ ]:
_synth = None

def get_synthesizer() -> SoundSynthesizer:
    """Lazy loader for the singleton synthesizer instance."""
    global _synth
    if _synth is None:
        _synth = SoundSynthesizer.get_instance()
    return _synth

# Gradio interface functions


In [ ]:
def compose_music(
    user_prompt: str,
    music_duration: int,
    creativity_level: float,
    prompt_adherence: float,
    seed_value: int
) -> Tuple[Optional[str], Optional[str]]:
    """
    Core pipeline function that orchestrates the music generation process.
    Called directly by the Gradio UI.

    Returns:
        Tuple containing (path_to_wav_file, status_message)
    """
    # Input validation
    if not user_prompt or not user_prompt.strip():
        return None, "कृपया एक संगीत प्रॉम्प्ट दर्ज करें।\nPlease enter a music prompt."

    if len(user_prompt) > 500:
        return None, "प्रॉम्प्ट बहुत लंबा है। कृपया 500 वर्णों तक सीमित रखें।\nPrompt too long. Please keep within 500 characters."

    try:
        # Get the global synthesizer
        synth_engine = get_synthesizer()

        # Adjust parameters based on user input
        temperature = creativity_level
        guidance = prompt_adherence

        # Generate raw audio
        audio_waveform, sample_rate = synth_engine.generate_audio_track(
            text_prompt=user_prompt,
            duration_sec=music_duration,
            temperature=temperature,
            guidance_scale=guidance,
            random_seed=seed_value if seed_value != -1 else None
        )

        # Save to temporary WAV file
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp_file:
            temp_path = tmp_file.name
            write_wav(temp_path, sample_rate, audio_waveform.astype(np.float32))

        # Clean up GPU memory
        synth_engine.clear_memory()

        success_msg = f"संगीत सफलतापूर्वक तैयार हुआ! | Music generated successfully!"
        return temp_path, success_msg

    except torch.cuda.OutOfMemoryError:
        return None, "GPU मेमोरी अपर्याप्त है। कृपया छोटी अवधि चुनें या CPU का उपयोग करें।\nGPU out of memory. Try shorter duration or use CPU."
    except Exception as error:
        return None, f" त्रुटि | Error: {str(error)}"

# Build the Gradio user interface


In [ ]:
def build_interface() -> gr.Blocks:
    """
    Construct the complete Gradio web interface with all controls,
    informational content, and audio output components.
    """
    with gr.Blocks(
        title="ध्वनि रचनाकार | Sound Composer",
        theme=gr.themes.Soft(primary_hue="emerald", neutral_hue="gray"),
        css="""
        .main-title {
            font-size: 2.2rem;
            text-align: center;
            background: linear-gradient(135deg, #2ecc71, #27ae60);
            -webkit-background-clip: text;
            background-clip: text;
            color: transparent;
            margin-bottom: 0.5rem;
        }
        .subtext {
            text-align: center;
            color: #666;
            margin-bottom: 1.5rem;
        }
        footer {
            text-align: center;
            margin-top: 2rem;
            color: #888;
            font-size: 0.85rem;
        }
        """
    ) as demo:
        # Header section
        gr.HTML("""
        <h1 class="main-title">ध्वनि रचनाकार | Sound Composer</h1>
        <p class="subtext">
             AI संगीत जनरेटर | Generate original music from ANY text description<br>
             Supports English, हिंदी, Hinglish, and any language prompt
        </p>
        """)

        # Main layout: two columns
        with gr.Row():
            # Left column: input controls
            with gr.Column(scale=2):
                prompt_input = gr.Textbox(
                    label="संगीत प्रॉम्प्ट | Music Prompt",
                    placeholder="उदा: एक शांत और सुखद धुन जो सुबह के समय बजती हो।\nExample: A peaceful and joyful melody that plays in the morning.",
                    lines=4,
                    elem_id="prompt-field"
                )

                with gr.Row():
                    duration_slider = gr.Slider(
                        minimum=3,
                        maximum=20,
                        value=10,
                        step=1,
                        label="अवधि (सेकंड) | Duration (seconds)",
                        info="लंबी अवधि के लिए अधिक समय लगेगा | Longer duration takes more time"
                    )

                with gr.Row():
                    creativity_slider = gr.Slider(
                        minimum=0.5,
                        maximum=2.0,
                        value=1.0,
                        step=0.05,
                        label="रचनात्मकता | Creativity",
                        info="उच्च मान = अधिक विविधता | Higher = more variety"
                    )

                    guidance_slider = gr.Slider(
                        minimum=1.0,
                        maximum=5.0,
                        value=3.0,
                        step=0.2,
                        label="प्रॉम्प्ट निष्ठा | Prompt adherence",
                        info="उच्च मान = प्रॉम्प्ट का अधिक पालन | Higher = follows prompt more strictly"
                    )

                seed_number = gr.Number(
                    label="यादृच्छिक बीज (वैकल्पिक) | Random Seed (optional)",
                    value=-1,
                    precision=0,
                    info="-1 = यादृच्छिक परिणाम | -1 = random result"
                )

                generate_button = gr.Button(
                    "संगीत बनाएँ | Generate Music",
                    variant="primary",
                    size="lg"
                )

            # Right column: output display
            with gr.Column(scale=2):
                audio_output = gr.Audio(
                    label="जेनरेटेड ऑडियो | Generated Audio",
                    type="filepath",
                    interactive=False
                )
                status_output = gr.Textbox(
                    label="स्थिति | Status",
                    lines=2,
                    interactive=False
                )

        # Example prompts
        gr.Markdown("""
        ### उदाहरण प्रॉम्प्ट | Example Prompts
        *Copy any of these to try different styles:*
        """)

        example_prompts = [
            "एक उदास पियानो धुन जो बारिश की आवाज़ के साथ मिलती है।"
            "Energetic rock guitar riff with drums and bass, upbeat and powerful.",
            "A calm Indian classical raga with sitar and tabla, peaceful evening mood.",
            "Ambient electronic music with ocean waves and soft synth pads, relaxing.",
            "एक रोमांटिक वायलिन और पियानो का युगल, प्रेम भरा एहसास।"
        ]

        examples_data = [[prompt, 10, 1.0, 3.0, -1] for prompt in example_prompts]

        gr.Examples(
            examples=examples_data,
            inputs=[prompt_input, duration_slider, creativity_slider, guidance_slider, seed_number],
            outputs=[audio_output, status_output],
            fn=compose_music,
            cache_examples=False
        )

        # Footer information
        gr.HTML("""
        <footer>
            <p>
                 संचालित | Powered by <strong>MusicGen</strong> (Meta AI) & Hugging Face Transformers<br>
                 GPU अनुशंसित | GPU recommended for faster generation (T4 or higher)
            </p>
        </footer>
        """)

        # Wire up the button click event
        generate_button.click(
            fn=compose_music,
            inputs=[prompt_input, duration_slider, creativity_slider, guidance_slider, seed_number],
            outputs=[audio_output, status_output]
        )

    return demo

# Entry point


In [ ]:
if __name__ == "__main__":
    print("=" * 60)
    print("ध्वनि रचनाकार - AI Music Generator")
    print("=" * 60)
    print("Starting Gradio web interface...")
    print("Access the interface at: http://localhost:7860")
    print("=" * 60)

    interface = build_interface()
    interface.launch(
        # Remove server_name and server_port to let Gradio find an available port automatically
        # server_name="0.0.0.0",
        # server_port=7860,
        share=True # Set share=True to generate a public link
    )